<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/Topo_CBP0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
import hashlib
import torch
import torch.nn as nn
import torch.nn.functional as F

class TopoCBPWorldModel(nn.Module):
    """
    Native PyTorch Topo-CBP Embodied World Model.
    Decomposes the manifold into an invariant prime coordinate bedrock
    and a Continual Backpropagation (CBP) plastic subspace.
    """
    def __init__(self, latent_dim: int = 128, action_dim: int = 6, hidden_dim: int = 256):
        super().__init__()
        self.latent_dim = latent_dim
        self.action_dim = action_dim
        self.hidden_dim = hidden_dim

        # Invariant Prime Coordinate Ring indices (P)
        self.prime_indices = torch.tensor([2, 3, 5, 7, 11, 13], dtype=torch.long)
        self.lambda_const = 0.9785142874

        # Native Network Layers
        self.in_proj = nn.Linear(latent_dim + action_dim, hidden_dim, bias=False)
        self.plastic1 = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.plastic2 = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj = nn.Linear(hidden_dim, latent_dim, bias=False)

        # Prime Anchor Reference Bedrock (Cached for exact bitwise restoration)
        with torch.no_grad():
            self.register_buffer("anchor_bedrock", self.in_proj.weight[self.prime_indices].clone())

        # Metabolic Utility Tracker (CBP)
        self.register_buffer("utility", torch.zeros(hidden_dim))
        self.decay = 0.99  # Running feature utility smoothing

    def forward(self, z_t: torch.Tensor, a_t: torch.Tensor) -> torch.Tensor:
        x = torch.cat([z_t, a_t], dim=-1)
        h = F.silu(self.in_proj(x))

        # Track neuron utility online (mean absolute activation across batch)
        if self.training:
            with torch.no_grad():
                current_util = h.abs().mean(dim=0)
                self.utility.mul_(self.decay).add_(current_util * (1.0 - self.decay))

        h1 = self.plastic1(h)
        h2 = F.silu(self.plastic2(h1))
        return self.out_proj(h2)

    # ==================== DUAL-PHASE TOPOLOGICAL GOVERNOR ====================

    def pre_optimizer_step(self):
        """Phase 1: Project gradients to zero across invariant prime anchors."""
        if self.in_proj.weight.grad is not None:
            self.in_proj.weight.grad[self.prime_indices] = 0.0

    def post_optimizer_step(self):
        """Phase 2: Bitwise register restoration against AdamW decoupled weight decay."""
        with torch.no_grad():
            self.in_proj.weight[self.prime_indices] = self.anchor_bedrock

    # ==================== ZERO-SHOCK METABOLIC CBP RECYCLING =================

    def recycle_dormant_units(self, replace_ratio: float = 0.04):
        """
        Audits running utility, recycles dormant plastic neurons,
        and protects the topological bedrock.
        """
        with torch.no_grad():
            num_replace = int(self.hidden_dim * replace_ratio)
            if num_replace == 0:
                return

            # Mask out prime indices so they are never targeted for metabolic reset
            eligible_utility = self.utility.clone()
            eligible_utility[self.prime_indices] = float("inf")

            # Identify dormant units
            _, dormant_indices = torch.topk(eligible_utility, k=num_replace, largest=False)

            # Incoming weights: fresh Kaiming normal initialization
            nn.init.kaiming_normal_(self.in_proj.weight[dormant_indices], nonlinearity="relu")

            # Outgoing weights: explicit zeroing (prevents downstream dynamic shock)
            self.plastic1.weight[:, dormant_indices] = 0.0

            # Reset utility ledger for refreshed units
            self.utility[dormant_indices] = eligible_utility[eligible_utility < float("inf")].mean()

    # ==================== CRYPTOGRAPHIC BARE-METAL INTERLOCK =================

    def compute_anchor_hash(self) -> str:
        """Returns the SHA-256 state hash of the prime coordinate ring."""
        raw_bytes = self.in_proj.weight[self.prime_indices].detach().cpu().contiguous().numpy().tobytes()
        return hashlib.sha256(raw_bytes).hexdigest()[:16]


# ==================== EXECUTION & DYNAMIC VERIFICATION ====================

# Initialize Model, Optimizer, and Hardware
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TopoCBPWorldModel().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)

initial_hash = model.compute_anchor_hash()
print(f"[INIT] Locked Anchor Hash: {initial_hash}")

# Streaming Lifelong Training Step
z = torch.randn(32, 128, device=device)
a = torch.randn(32, 6, device=device)
z_next = torch.randn(32, 128, device=device)

# Forward & Backward Pass
pred_next = model(z, a)
loss = F.mse_loss(pred_next, z_next)

optimizer.zero_grad()
loss.backward()

# 1. Dual-Phase Topological Control (Pre)
model.pre_optimizer_step()

# Standard Optimizer Step
optimizer.step()

# 2. Dual-Phase Topological Control (Post)
model.post_optimizer_step()

# 3. Metabolic Capacity Renewal (Online)
model.recycle_dormant_units(replace_ratio=0.04)

# Dynamic Verification & Telemetry
with torch.no_grad():
    current_anchors = model.in_proj.weight[model.prime_indices]
    drift = torch.norm(current_anchors - model.anchor_bedrock).item()

current_hash = model.compute_anchor_hash()
status = "LOCKED" if (current_hash == initial_hash and drift == 0.0) else "CORRUPTED"

# Hardware Interlock Check
assert status == "LOCKED", f"Safety Interlock Tripped: Coordinate Shear Detected! Drift: {drift:.10e}"

print(f"[STEP 1] Loss: {loss.item():.6f}")
print(f"[VERIFY] Anchor Hash: {current_hash} (Status: {status} | Measured Drift: {drift:.10f})")

[INIT] Locked Anchor Hash: f056d3f3b5f6d44a
[STEP 1] Loss: 0.999131
[VERIFY] Anchor Hash: f056d3f3b5f6d44a (Status: LOCKED | Measured Drift: 0.0000000000)
